**1. Mounting Google Drive in Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Install library**

In [ ]:
!pip install opencv-python numpy pandas
!pip install skrebate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 821.6 kB/s eta 0:00:00


**3. Set input/output paths and ensure features folder exists**

In [ ]:
# Update this path according to the location of the dataset
base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'

# Input folder containing segmented images
segmentation_folder = os.path.join(
    base_folder,
    'Segmentation'
)

# Output folder for feature extraction results
features_folder = os.path.join(
    base_folder,
    'Features'
)

# Create output folder if it does not exist
os.makedirs(
    features_folder,
    exist_ok=True
)

**4. Feature Extraction**

**- Morphological**

In [ ]:
import os
import numpy as np
import pandas as pd

# Base folder
base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'

features_folder = os.path.join(base_folder, 'Features')

# Input and output paths
input_path = os.path.join(features_folder, 'Morphological_Raw.xlsx')
output_path = os.path.join(features_folder,'Calculate_Morphological.csv')

def load_data(input_path):
    df = pd.read_excel(input_path)
    return df

def calculate_morphological_features(df):
    # Shape Index = (Width / Length) × 100
    df["Shape_Index"] = (
        df["Width"] / df["Length"]
    ) * 100

    # Eccentricity = sqrt((Length/2)^2 - (Width/2)^2) / (Length/2)
    df["Eccentricity"] = (
        np.sqrt(
            (df["Length"] / 2) ** 2 -
            (df["Width"] / 2) ** 2
        )
        / (df["Length"] / 2)
    )
    return df

def save_data(df, output_path):
    df.to_csv(output_path, index=False)

def main():
    # Load raw morphological measurements
    df = load_data(input_path)
    # Calculate morphological features
    df = calculate_morphological_features(df)
    # Save calculated morphological features
    save_data(df, output_path)
main()

**- Embryo_Vascular1: Perimeter, Sinus Terminalis Area, Embryo Vascular Area, and Fractal Dimension**

In [ ]:
import os
import cv2 as cv
import numpy as np
import pandas as pd

# Load Binary Image
def load_binary_image(img_path):
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    if img is None:
        return None
    _, binary = cv.threshold(
        img,
        0,
        255,
        cv.THRESH_BINARY + cv.THRESH_OTSU
    )
    return binary

# Perimeter
def calculate_perimeter(binary):
    contours, _ = cv.findContours(
        binary,
        cv.RETR_EXTERNAL,
        cv.CHAIN_APPROX_SIMPLE
    )
    perimeter = 0
    for cnt in contours:
        perimeter += cv.arcLength(cnt, True)
    return perimeter

# Sinus Terminalis Area
def calculate_sinus_terminalis_area(binary):
    area = np.sum(binary > 0)
    return area

# Embryo Vascular Area
def calculate_embryo_vascular_area(binary):
    contours, _ = cv.findContours(
        binary,
        cv.RETR_EXTERNAL,
        cv.CHAIN_APPROX_SIMPLE
    )
    if len(contours) == 0:
        return 0
    largest = max(contours, key=cv.contourArea)
    hull = cv.convexHull(largest)
    hull_area = cv.contourArea(hull)
    return hull_area


# Box-counting-based fractal dimension using intensity variation
def calculate_fractal_dimension(binary, min_box_size=2, max_box_size=64):
    img = binary / 255.0
    sizes = []
    counts = []
    for box_size in range(min_box_size, max_box_size, 2):
        S = box_size
        count = 0
        for i in range(0, img.shape[0], S):
            for j in range(0, img.shape[1], S):
                patch = img[i:i+S, j:j+S]
                if patch.size == 0:
                    continue
                min_val = patch.min()
                max_val = patch.max()
                if max_val > min_val:
                    count += 1
        if count > 0:
            sizes.append(np.log(1.0/S))
            counts.append(np.log(count))
    if len(sizes) < 2:
        return 0
    coeffs = np.polyfit(sizes, counts, 1)
    return coeffs[0]

# Feature extraction
def extract_features_from_folder(input_folder, output_csv):
    data = []
    for fname in os.listdir(input_folder):
        if fname.lower().endswith(('.png')):
            fpath = os.path.join(input_folder, fname)
            binary = load_binary_image(fpath)
            if binary is None:
                continue
            perimeter = calculate_perimeter(binary)
            sinus_area = calculate_sinus_terminalis_area(binary)
            vascular_area = calculate_embryo_vascular_area(binary)
            fractal_dim = calculate_fractal_dimension(binary)

            data.append({
                "Image_Number": os.path.splitext(fname)[0],
                "Perimeter": perimeter,
                "Sinus_Terminalis_Area": sinus_area,
                "Embryo_Vascular_Area": vascular_area,
                "Fractal_Dimension": fractal_dim
            })

    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)

# Execution
if __name__ == '__main__':
    input_folder = os.path.join(segmentation_folder, 'Masking_Erosion_Day-5')
    output_csv = os.path.join(features_folder, 'Embryo_Vascular1.csv')
    extract_features_from_folder(input_folder, output_csv)

**- Embryo_Vascular2: Branch Nodes**

In [ ]:
import cv2 as cv
import numpy as np
import pandas as pd
import os
from skimage.morphology import skeletonize

# Branch Node Extraction
def process_image(input_folder):
    img = cv.imread(input_folder, cv.IMREAD_GRAYSCALE)

    # Thresholded-otsu
    _, thresh = cv.threshold(img, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    # Skeletonize
    skeleton = skeletonize(thresh > 0)

    # Branch nodes detection
    nodes = []
    for y in range(1, skeleton.shape[0]-1):
        for x in range(1, skeleton.shape[1]-1):
            if skeleton[y, x]:
                neighbors = np.sum(skeleton[y-1:y+2, x-1:x+2]) - 1
                # Branch nodes criterion
                if neighbors >= 3:
                    nodes.append((x,y))

    num_branches = len(nodes)
    return num_branches

def process_all_images(input_folder, output_csv):
    results = []
    for fname in os.listdir(input_folder):
        if fname.lower().endswith(('.png')):
            fpath = os.path.join(input_folder, fname)
            num_branches = process_image(fpath)
            clean_name = os.path.splitext(fname)[0]
            results.append({"Image_Number": clean_name, "Branch_Nodes": num_branches})

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    return df

if __name__ == '__main__':
    input_folder = os.path.join(segmentation_folder, 'Frangi_Vessel_Visibility_Day-5')
    output_csv = os.path.join(features_folder, 'Embryo_Vascular2.csv')
    df = process_all_images(input_folder, output_csv)

**- Textural GLCM: Energy, Contrast, Correlation, Homogeneity, Entropy**

In [ ]:
import os
import cv2 as cv
import numpy as np
import pandas as pd
from glob import glob
from skimage.feature import graycomatrix, graycoprops

from IPython.display import display


def extract_glcm_features(img_path):
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    if img is None:
        return None
    # Quantization to 8 gray levels (0-7)
    img_quantized = (img / 32).astype(np.uint8)

    # GLCM, distance = 1 pixel, angles = 0°, 45°, 90°, 135°
    glcm = graycomatrix(
        img_quantized,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=8,
        symmetric=True,
        normed=True
    )

    # GLCM Features
    energy = np.mean(graycoprops(glcm, 'energy'))
    contrast = np.mean(graycoprops(glcm, 'contrast'))
    correlation = np.mean(graycoprops(glcm, 'correlation'))
    homogeneity = np.mean(graycoprops(glcm, 'homogeneity'))

    entropy_list = []
    for angle in range(glcm.shape[3]):
        P = glcm[:, :, 0, angle]
        entropy = -np.sum(P * np.log2(P + 1e-10))
        entropy_list.append(entropy)
    entropy = np.mean(entropy_list)

    return (
        energy,
        contrast,
        correlation,
        homogeneity,
        entropy
    )


def main():
    # Process all images
    results = []
    image_paths = sorted(glob(os.path.join(image_folder, '*.png')))

    print(f'Total images: {len(image_paths)}')
    for img_path in image_paths:
        features = extract_glcm_features(img_path)
        if features is None:
            continue
        energy, contrast, correlation, homogeneity, entropy = features
        results.append([
            os.path.splitext(os.path.basename(img_path))[0],
            energy,
            contrast,
            correlation,
            homogeneity,
            entropy
        ])

    # Dataframe
    df = pd.DataFrame(
        results,
        columns=[
            'Image_Number',
            'GLCM_Energy',
            'GLCM_Contrast',
            'GLCM_Correlation',
            'GLCM_Homogeneity',
            'GLCM_Entropy'
        ]
    )

    df.to_csv(output_csv, index=False)

# Execution
if __name__ == "__main__":
    image_folder = os.path.join(segmentation_folder, 'Masking_Erosion_Day-5')
    output_csv = os.path.join(features_folder, 'Textural_GLCM.csv')
    main()

**5. Hybrid Feature Fusion**

In [ ]:
import pandas as pd
import os

# File paths
base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'
features_folder = os.path.join(base_folder, 'Features')

# Input feature datasets
morphological_csv = os.path.join(features_folder, 'Calculate_Morphological.csv')
vascular1_csv = os.path.join(features_folder, 'Embryo_Vascular1.csv')
vascular2_csv = os.path.join(features_folder, 'Embryo_Vascular2.csv')
glcm_raw_csv = os.path.join(features_folder, 'GLCM.csv')
groundtruth_csv = os.path.join(features_folder, 'GroundTruth_VentSexing.csv')


# Load datasets
def load_datasets(morphological_csv, vascular1_csv, vascular2_csv, glcm_csv, groundtruth_csv):
    morphological = pd.read_csv(morphological_csv)
    vascular1 = pd.read_csv(vascular1_csv)
    vascular2 = pd.read_csv(vascular2_csv)
    glcm = pd.read_csv(glcm_csv)
    groundtruth = pd.read_csv(groundtruth_csv)

    return (morphological, vascular1, vascular2, glcm, groundtruth)

# Verify corresponding image order
def verify_image_order(morphological, vascular1, vascular2, glcm, groundtruth):
    image_order = morphological[
        "Image_Number"
    ]
    assert image_order.equals(
        vascular1["Image_Number"]
    ), "Image order mismatch: Morphological vs Vascular1"

    assert image_order.equals(
        vascular2["Image_Number"]
    ), "Image order mismatch: Morphological vs Vascular2"

    assert image_order.equals(
        glcm["Image_Number"]
    ), "Image order mismatch: Morphological vs GLCM"

    assert image_order.equals(
        groundtruth["Image_Number"]
    ), "Image order mismatch: Morphological vs GroundTruth"

    print(
        "\nImage order verification: PASSED"
    )

# Select feature columns
def select_feature_columns(morphological, vascular1, vascular2, glcm):
    # Morphological
    morphological_features = morphological[
        [
            "Length",
            "Width",
            "Shape_Index",
            "Weight",
            "Eccentricity"
        ]
    ]

    # Embryologic Vascular1
    vascular_features = vascular1[
        [
            "Perimeter",
            "Sinus_Terminalis_Area",
            "Embryo_Vascular_Area",
            "Fractal_Dimension"
        ]
    ]

    # Embryologic Vascular2
    branch_features = vascular2[
        [
            "Branch_Nodes"
        ]
    ]

    # Column-wise concatenation
    # Combine Vascular1 and Vascular2
    vascular_features = pd.concat(
        [
            vascular_features.reset_index(drop=True),
            branch_features.reset_index(drop=True)
        ],
        axis=1
    )

    # Textural GLCM
    glcm_features = glcm[
        [
            "GLCM_Energy",
            "GLCM_Contrast",
            "GLCM_Correlation",
            "GLCM_Homogeneity",
            "GLCM_Entropy"
        ]
    ]

    return (
        morphological_features,
        vascular_features,
        glcm_features
    )

# Create Morphological dataset with Class
def create_morphological_dataset(
    morphological_csv,
    groundtruth_csv,
    features_folder
):

    morphological = pd.read_csv(morphological_csv)
    groundtruth = pd.read_csv(groundtruth_csv)

    # Verify corresponding image order
    assert morphological[
        "Image_Number"
    ].equals(
        groundtruth["Image_Number"]
    ), "Image order mismatch: Morphological vs GroundTruth"

    # Select morphological features
    morphological_features = morphological[
        [
            "Image_Number",
            "Length",
            "Width",
            "Shape_Index",
            "Weight",
            "Eccentricity"
        ]
    ]

    # Column-wise concatenation
    morphological_dataset = pd.concat(
        [
            morphological_features.reset_index(drop=True),
            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    morphological_feature_columns = [
        "Length",
        "Width",
        "Shape_Index",
        "Weight",
        "Eccentricity"
    ]

    assert len(morphological_feature_columns) == 5

    expected_columns = (
        ["Image_Number"]
        + morphological_feature_columns
        + ["Class"]
    )

    assert (
        morphological_dataset.columns.tolist()
        == expected_columns
    )

    # 1. Save dataset:  Morphological
    output_csv = os.path.join(features_folder, "Morphological.csv")

    morphological_dataset.to_csv(
        output_csv,
        index=False
    )

    return morphological_dataset

# Embryologic Vascular
def create_embryologic_vascular_dataset(
    vascular1,
    vascular_features,
    groundtruth,
    features_folder
):

    # Column-wise concatenation
    embryologic_vascular_dataset = pd.concat(
        [
            vascular1[
                "Image_Number"
            ].reset_index(drop=True),

            vascular_features.reset_index(drop=True),

            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate Vascular
    vascular_feature_columns = [
        "Perimeter",
        "Sinus_Terminalis_Area",
        "Embryo_Vascular_Area",
        "Fractal_Dimension",
        "Branch_Nodes"
    ]

    assert len(vascular_feature_columns) == 5

    expected_columns = (
        ["Image_Number"]
        + vascular_feature_columns
        + ["Class"]
    )

    assert (
        embryologic_vascular_dataset.columns.tolist()
        == expected_columns
    )

    # 2. Save dataset: Embryologic Vascular
    output_csv = os.path.join(features_folder, "Embryologic_Vascular.csv")
    embryologic_vascular_dataset.to_csv(
        output_csv,
        index=False
    )

    return embryologic_vascular_dataset

# Create Textural GLCM dataset with Class
def create_glcm_dataset(
    glcm_raw_csv,
    groundtruth_csv,
    features_folder
):

    glcm = pd.read_csv(glcm_raw_csv)
    groundtruth = pd.read_csv(groundtruth_csv)

    # Verify corresponding image order
    assert glcm[
        "Image_Number"
    ].equals(
        groundtruth["Image_Number"]
    ), "Image order mismatch: GLCM vs GroundTruth"

    # Select GLCM features
    glcm_features = glcm[
        [
            "Image_Number",
            "GLCM_Energy",
            "GLCM_Contrast",
            "GLCM_Correlation",
            "GLCM_Homogeneity",
            "GLCM_Entropy"
        ]
    ]

    # Column-wise concatenation
    glcm_dataset = pd.concat(
        [
            glcm_features.reset_index(drop=True),
            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    glcm_feature_columns = [
        "GLCM_Energy",
        "GLCM_Contrast",
        "GLCM_Correlation",
        "GLCM_Homogeneity",
        "GLCM_Entropy"
    ]

    assert len(glcm_feature_columns) == 5

    expected_columns = (
        ["Image_Number"]
        + glcm_feature_columns
        + ["Class"]
    )

    assert (
        glcm_dataset.columns.tolist()
        == expected_columns
    )

    # 3. Save dataset: Textural (GLCM)
    output_csv = os.path.join(features_folder, "Textural_GLCM.csv")
    glcm_dataset.to_csv(
        output_csv,
        index=False
    )
    return glcm_dataset

# Morphological + Embryologic Vascular
def create_morphological_vascular_dataset(
    morphological,
    morphological_features,
    vascular_features,
    groundtruth,
    features_folder
):

    # Column-wise concatenation
    morphological_vascular_features = pd.concat(
        [
            morphological_features.reset_index(drop=True),
            vascular_features.reset_index(drop=True)
        ],
        axis=1
    )

    morphological_vascular_dataset = pd.concat(
        [
            morphological[
                "Image_Number"
            ].reset_index(drop=True),

            morphological_vascular_features,

            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    morphological_vascular_feature_columns = [
        "Length",
        "Width",
        "Shape_Index",
        "Weight",
        "Eccentricity",
        "Perimeter",
        "Sinus_Terminalis_Area",
        "Embryo_Vascular_Area",
        "Fractal_Dimension",
        "Branch_Nodes"
    ]

    assert len(
        morphological_vascular_feature_columns
    ) == 10

    expected_columns = (
        ["Image_Number"]
        + morphological_vascular_feature_columns
        + ["Class"]
    )

    assert (
        morphological_vascular_dataset.columns.tolist()
        == expected_columns
    )

    # 4. Save dataset: Morphological + Embryologic Vascular
    output_csv = os.path.join(features_folder, "Morphological_Embryologic_Vascular.csv")
    morphological_vascular_dataset.to_csv(
        output_csv,
        index=False
    )

    return morphological_vascular_dataset

# Morphological + Textural (GLCM)
def create_morphological_glcm_dataset(
    morphological,
    morphological_features,
    glcm_features,
    groundtruth,
    features_folder
):

    # Column-wise concatenation
    morphological_glcm_features = pd.concat(
        [
            morphological_features.reset_index(drop=True),
            glcm_features.reset_index(drop=True)
        ],
        axis=1
    )

    morphological_glcm_dataset = pd.concat(
        [
            morphological[
                "Image_Number"
            ].reset_index(drop=True),

            morphological_glcm_features,

            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    morphological_glcm_feature_columns = [
        "Length",
        "Width",
        "Shape_Index",
        "Weight",
        "Eccentricity",
        "GLCM_Energy",
        "GLCM_Contrast",
        "GLCM_Correlation",
        "GLCM_Homogeneity",
        "GLCM_Entropy"
    ]

    assert len(
        morphological_glcm_feature_columns
    ) == 10

    expected_columns = (
        ["Image_Number"]
        + morphological_glcm_feature_columns
        + ["Class"]
    )

    assert (
        morphological_glcm_dataset.columns.tolist()
        == expected_columns
    )

    # 5. Save dataset: Morphological + Textural (GLCM)
    output_csv = os.path.join(features_folder, "Morphological_Textural_GLCM.csv")
    morphological_glcm_dataset.to_csv(
        output_csv,
        index=False
    )
    return morphological_glcm_dataset

# Embryologic Vascular + Textural (GLCM)
def create_vascular_glcm_dataset(
    vascular1,
    vascular_features,
    glcm_features,
    groundtruth,
    features_folder
):

    # Column-wise concatenation
    vascular_glcm_features = pd.concat(
        [
            vascular_features.reset_index(drop=True),
            glcm_features.reset_index(drop=True)
        ],
        axis=1
    )

    vascular_glcm_dataset = pd.concat(
        [
            vascular1[
                "Image_Number"
            ].reset_index(drop=True),

            vascular_glcm_features,

            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    vascular_glcm_feature_columns = [
        "Perimeter",
        "Sinus_Terminalis_Area",
        "Embryo_Vascular_Area",
        "Fractal_Dimension",
        "Branch_Nodes",
        "GLCM_Energy",
        "GLCM_Contrast",
        "GLCM_Correlation",
        "GLCM_Homogeneity",
        "GLCM_Entropy"
    ]

    assert len(
        vascular_glcm_feature_columns
    ) == 10

    expected_columns = (
        ["Image_Number"]
        + vascular_glcm_feature_columns
        + ["Class"]
    )

    assert (
        vascular_glcm_dataset.columns.tolist()
        == expected_columns
    )

    # 6. Save dataset: Embryologic Vascular + Textural (GLCM)
    output_csv = os.path.join(features_folder, "Embryologic_Vascular_Textural_GLCM.csv")
    vascular_glcm_dataset.to_csv(
        output_csv,
        index=False
    )
    return vascular_glcm_dataset

# Morphological + Embryologic Vascular + Textural (GLCM)
def create_full_hybrid_dataset(
    morphological,
    morphological_features,
    vascular_features,
    glcm_features,
    groundtruth,
    features_folder
):

    # Column-wise concatenation
    full_hybrid_features = pd.concat(
        [
            morphological_features.reset_index(drop=True),
            vascular_features.reset_index(drop=True),
            glcm_features.reset_index(drop=True)
        ],
        axis=1
    )

    full_hybrid_dataset = pd.concat(
        [
            morphological[
                "Image_Number"
            ].reset_index(drop=True),

            full_hybrid_features,

            groundtruth[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    full_hybrid_feature_columns = [
        "Length",
        "Width",
        "Shape_Index",
        "Weight",
        "Eccentricity",
        "Perimeter",
        "Sinus_Terminalis_Area",
        "Embryo_Vascular_Area",
        "Fractal_Dimension",
        "Branch_Nodes",
        "GLCM_Energy",
        "GLCM_Contrast",
        "GLCM_Correlation",
        "GLCM_Homogeneity",
        "GLCM_Entropy"
    ]

    assert len(
        full_hybrid_feature_columns
    ) == 15

    expected_columns = (
        ["Image_Number"]
        + full_hybrid_feature_columns
        + ["Class"]
    )

    assert (
        full_hybrid_dataset.columns.tolist()
        == expected_columns
    )

    # 7. Save dataset: Morphological + Embryologic Vascular + Textural (GLCM)
    output_csv = os.path.join(features_folder, "Morphological_Embryologic_Vascular_Textural_GLCM.csv")
    full_hybrid_dataset.to_csv(
        output_csv,
        index=False
    )
    return full_hybrid_dataset

# Main process
def main():

    # Load datasets
    (
        morphological,
        vascular1,
        vascular2,
        glcm,
        groundtruth
    ) = load_datasets(
        morphological_csv,
        vascular1_csv,
        vascular2_csv,
        glcm_raw_csv,
        groundtruth_csv
    )

    # Verify corresponding image order
    verify_image_order(
        morphological,
        vascular1,
        vascular2,
        glcm,
        groundtruth
    )

    # Select feature columns
    (
        morphological_features,
        vascular_features,
        glcm_features
    ) = select_feature_columns(
        morphological,
        vascular1,
        vascular2,
        glcm
    )

    # 1. Save dataset: Morphological
    create_morphological_dataset(
        morphological_csv,
        groundtruth_csv,
        features_folder
    )

    # 2. Save dataset: Embryologic Vascular
    embryologic_vascular_dataset = (
        create_embryologic_vascular_dataset(
            vascular1,
            vascular_features,
            groundtruth,
            features_folder
        )
    )

    # 3. Save dataset: Textural (GLCM)
    glcm_dataset = create_glcm_dataset(
        glcm_raw_csv,
        groundtruth_csv,
        features_folder
    )

    # 4. Save dataset: Morphological + Embryologic Vascular
    morphological_vascular_dataset = (
        create_morphological_vascular_dataset(
            morphological,
            morphological_features,
            vascular_features,
            groundtruth,
            features_folder
        )
    )

    # 5. Save dataset: Morphological + Textural (GLCM)
    morphological_glcm_dataset = (
        create_morphological_glcm_dataset(
            morphological,
            morphological_features,
            glcm_features,
            groundtruth,
            features_folder
        )
    )

    # 6. Save dataset: Embryologic Vascular + Textural (GLCM)
    vascular_glcm_dataset = (
        create_vascular_glcm_dataset(
            vascular1,
            vascular_features,
            glcm_features,
            groundtruth,
            features_folder
        )
    )

    # 7. Save dataset: Morphological + Embryologic Vascular + Textural (GLCM)
    full_hybrid_dataset = (
        create_full_hybrid_dataset(
            morphological,
            morphological_features,
            vascular_features,
            glcm_features,
            groundtruth,
            features_folder
        )
    )

# Run feature fusion
if __name__ == '__main__':
    main()

**6. Hybrid Feature Fusion: Acquisition_Stage.csv & Morphological_Embryologic_Vascular_Textural_GLCM.csv**

In [ ]:
import pandas as pd
import os

base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'
features_folder = os.path.join(base_folder, 'Features')

# Input datasets
acquisition_stage_csv = os.path.join(
    features_folder,
    'Acquisition_Stage.csv'
)
hybrid_csv = os.path.join(
    features_folder,
    'Morphological_Embryologic_Vascular_Textural_GLCM.csv'
)

def load_datasets(
    acquisition_stage_csv,
    hybrid_csv
):
    acquisition_stage = pd.read_csv(
        acquisition_stage_csv
    )
    hybrid = pd.read_csv(
        hybrid_csv
    )
    return (
        acquisition_stage,
        hybrid
    )

# Verify corresponding image order
def verify_image_order(
    acquisition_stage,
    hybrid
):
    image_order = hybrid[
        "Image_Number"
    ]
    assert image_order.equals(
        acquisition_stage["Image_Number"]
    ), (
        "Image order mismatch: "
        "Hybrid vs Acquisition Stage"
    )

# Select acquisition stage
def select_acquisition_stage(
    acquisition_stage
):
    acquisition_stage_features = acquisition_stage[
        [
            "Acquisition_Stage"
        ]
    ]
    return acquisition_stage_features

# Selct hybrid features
def select_hybrid_features(
    hybrid
):
    # Morphological
    morphological_features = hybrid[
        [
            "Length",
            "Width",
            "Shape_Index",
            "Weight",
            "Eccentricity"
        ]
    ]

    # Embryologic Vascular
    vascular_features = hybrid[
        [
            "Perimeter",
            "Sinus_Terminalis_Area",
            "Embryo_Vascular_Area",
            "Fractal_Dimension",
            "Branch_Nodes"
        ]
    ]

    # Textural GLCM
    glcm_features = hybrid[
        [
            "GLCM_Energy",
            "GLCM_Contrast",
            "GLCM_Correlation",
            "GLCM_Homogeneity",
            "GLCM_Entropy"
        ]
    ]

    return (
        morphological_features,
        vascular_features,
        glcm_features
    )

def create_full_hybrid_dataset(
    hybrid,
    acquisition_stage_features,
    morphological_features,
    vascular_features,
    glcm_features,
    features_folder
):

    # Column-wise concatenation
    full_hybrid_features = pd.concat(
        [
            acquisition_stage_features.reset_index(
                drop=True
            ),
            morphological_features.reset_index(
                drop=True
            ),
            vascular_features.reset_index(
                drop=True
            ),
            glcm_features.reset_index(
                drop=True
            )
        ],
        axis=1
    )

    # Add Image_Number and Class
    full_hybrid_dataset = pd.concat(
        [
            hybrid[
                "Image_Number"
            ].reset_index(drop=True),

            full_hybrid_features,

            hybrid[
                "Class"
            ].reset_index(drop=True)
        ],
        axis=1
    )

    # Validate features
    full_hybrid_feature_columns = [
        "Acquisition_Stage",

        # Morphological
        "Length",
        "Width",
        "Shape_Index",
        "Weight",
        "Eccentricity",

        # Embryologic Vascular
        "Perimeter",
        "Sinus_Terminalis_Area",
        "Embryo_Vascular_Area",
        "Fractal_Dimension",
        "Branch_Nodes",

        # Textural GLCM
        "GLCM_Energy",
        "GLCM_Contrast",
        "GLCM_Correlation",
        "GLCM_Homogeneity",
        "GLCM_Entropy"
    ]

    assert len(
        full_hybrid_feature_columns
    ) == 16

    # Expected column order
    expected_columns = (
        [
            "Image_Number"
        ]
        + full_hybrid_feature_columns
        + [
            "Class"
        ]
    )

    assert (
        full_hybrid_dataset.columns.tolist()
        == expected_columns
    )

    # Save dataset
    output_csv = os.path.join(features_folder, "Hybrid_Feature.csv")
    full_hybrid_dataset.to_csv(
        output_csv,
        index=False
    )
    return full_hybrid_dataset

def main():
    (
        acquisition_stage,
        hybrid
    ) = load_datasets(
        acquisition_stage_csv,
        hybrid_csv
    )

    # Verify corresponding image order
    verify_image_order(
        acquisition_stage,
        hybrid
    )

    # Select Acquisition Stage
    acquisition_stage_features = (
        select_acquisition_stage(
            acquisition_stage
        )
    )

    # Select Hybrid Features
    (
        morphological_features,
        vascular_features,
        glcm_features
    ) = select_hybrid_features(
        hybrid
    )

    # Create Final Hybrid Dataset
    full_hybrid_dataset = (
        create_full_hybrid_dataset(
            hybrid,
            acquisition_stage_features,
            morphological_features,
            vascular_features,
            glcm_features,
            features_folder
        )
    )

if __name__ == '__main__':
    main()

**7. Feature Importance Ranking**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from skrebate import ReliefF
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

# Baca dataset
df = pd.read_csv(
    '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs/Features/Morphological_Embryologic_Vascular_Textural_GLCM.csv'
)

# Pisahkan fitur dan target
X = df.drop(['Class', 'Image_Number'], axis=1)
y = df['Class'].map({'M': 1, 'F': 0}).astype(int)

# Hilangkan prefix "GLCM_" dari nama kolom
X.columns = X.columns.str.replace(r'^GLCM_', '', regex=True)

# 3. Split data: 80% Train+Validation dan 20% Test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=100,
    stratify=y,
    random_state=42
)

# Split Train+Validation menjadi:
# Training = 353
# Validation = 50
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=50,
    stratify=y_trainval,
    random_state=42
)

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)
print(f"Training   : {len(X_train)} samples")
print(f"Validation : {len(X_val)} samples")
print(f"Testing    : {len(X_test)} samples")
print(f"Total      : {len(X)} samples")

# Distribusi kelas pada setiap subset
print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

print("\nTraining:")
print(y_train.map({0: 'Female', 1: 'Male'}).value_counts())
print("\nValidation:")
print(y_val.map({0: 'Female', 1: 'Male'}).value_counts())
print("\nTesting:")
print(y_test.map({0: 'Female', 1: 'Male'}).value_counts())

# ReliefF Feature Importance
relief = ReliefF(
    n_neighbors=100,
    n_features_to_select=X_train.shape[1]
)

relief.fit(
    X_train.values,
    y_train.values
)

# Buat ranking feature importance
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,  # sudah tanpa prefix GLCM_
    'Importance': relief.feature_importances_
})

# Urutkan dari importance tertinggi
feature_importances = (
    feature_importances
    .sort_values(by='Importance', ascending=False)
    .reset_index(drop=True)
)

# Pembulatan untuk tampilan
feature_importances['Importance'] = feature_importances['Importance'].round(3)

print("\n" + "=" * 60)
print("RELIEFF FEATURE IMPORTANCE RANKING")
print("=" * 60)
print(feature_importances.to_string(index=False))

# Simpan ranking feature importance
feature_importances.to_csv(
    '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs/Features/ReliefF_Feature_Importance.csv',
    index=False
)
print("\nFeature importance ranking saved.")

# Visualisasi Top 15 Feature Importance
top_n = 15
feature_importances_sorted = feature_importances.head(top_n).copy()
plt.figure(figsize=(6, 4))

ax = sns.barplot(
    x='Importance',
    y='Feature',
    hue='Feature',
    data=feature_importances_sorted,
    palette='viridis',
    legend=False
)
#plt.title(f'ReliefF Feature Importance Ranking', fontsize=10)
plt.xlabel('Importance Score', fontsize=9)
plt.ylabel('Feature', fontsize=9)
# Format angka pada sumbu X
ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))

# Tambahkan label angka di ujung balok (sebelah kanan)
for i, imp in enumerate(feature_importances_sorted['Importance']):
    ax.text(imp + 0.001, i, f'{imp:.3f}', va='center', fontsize=8)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)

plt.tight_layout()
plt.show()